In [17]:
!pip install wikipedia-api neo4j


In [18]:
!pip install openai==0.28

In [19]:
import wikipediaapi
import openai
from neo4j import GraphDatabase
import re


In [35]:
openai.api_key = "your open ai secrect key"

In [21]:
def extract_entities(text):
    prompt = f"Extract entities and their types from the following text:\n\n{text}\n\nFormat: Entity - Type (e.g., John Doe - Person)"
    response = openai.ChatCompletion.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return response["choices"][0]["message"]["content"]

In [22]:
uri = "bolt://44.201.29.189:7687"
username = "neo4j"
password = "cameras-oxides-vector"
driver = GraphDatabase.driver(uri, auth=(username, password))

In [23]:
def create_nodes(tx, label, name):
    query = f"MERGE (n:{label} {{name: $name}})"
    tx.run(query, name=name)

def create_relationship(tx, entity1, entity2, relationship):
    query = (
        f"MATCH (a:{entity1}), (b:{entity2}) "
        f"WHERE a.name = $entity1 AND b.name = $entity2 "
        f"MERGE (a)-[r:{relationship}]->(b) "
        f"RETURN type(r)"
    )
    tx.run(query, entity1=entity1, entity2=entity2)

In [24]:
wiki = wikipediaapi.Wikipedia(language='en', user_agent='my_bot (abc@example.com)')
page_title = "Artificial intelligence"
page = wiki.page(page_title)

if page.exists():
    wiki_text = page.text
    print(f"Text from {page_title}:\n{wiki_text[:1000]}")
else:
    print(f"Page '{page_title}' does not exist.")

Text from Artificial intelligence:
Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.
High-profile applications of AI include advanced web search engines (e.g., Google Search); recommendation systems (used by YouTube, Amazon, and Netflix); virtual assistants (e.g., Google Assistant, Siri, and Alexa); autonomous vehicles (e.g., Waymo); generative and creative tools (e.g., language models and AI art); and superhuman play and analysis in strategy games (e.g., chess and Go). However, many AI applications are not perceived as AI: "A lot of cutting edge AI has filtered into general appl

In [25]:
entities_text = extract_entities(wiki_text[:2000])
print("Extracted Entities:\n", entities_text)

Extracted Entities:
 Artificial intelligence - Concept
AI - Acronym/Concept
computational systems - Concept
learning - Research area
reasoning - Research area
problem-solving - Research area
perception - Research area
decision-making - Research area
computer science - Field
methods and software - Concept
machines - Concept
web search engines - Product category
Google Search - Product
recommendation systems - System type
YouTube - Organization
Amazon - Organization
Netflix - Organization
virtual assistants - Product category
Google Assistant - Product
Siri - Product
Alexa - Product
autonomous vehicles - Technology category
Waymo - Organization
generative and creative tools - Technology category
language models - Technology
AI art - Concept
chess - Game
Go - Game
knowledge representation - Research area
planning - Research area
natural language processing - Research area
support for robotics - Research area
search (technique) - Method
mathematical optimization - Method
formal logic - Met

In [29]:
import re

def clean_label(entity_type: str) -> str:
    """Convert entity_type like 'Organization/Company' -> ['Organization','Company']"""
    parts = [p for p in re.split(r'[^A-Za-z0-9]+', entity_type or '') if p]
    return parts or ['Entity']

def create_node(tx, entity_type, entity_name):
    labels = ':' + ':'.join(clean_label(entity_type))
    query = f"MERGE (e{labels} {{name: $name}}) RETURN e"
    tx.run(query, name=entity_name)


def clean_entity_name(entity_name: str) -> str:
    """Remove numbering like '1. Entity' and trim whitespace."""
    return re.sub(r"^\d+\.\s*", "", entity_name).strip()

def parse_entities(entities_text: str):
    """
    Parse lines formatted as:
      Entity Name - Entity Type
    Returns: List[(entity_name, entity_type)]
    """
    entity_pattern = r"^(.+?)\s*-\s*(.+)$"
    entities = []
    for line in entities_text.splitlines():
        line = line.strip()
        if not line:
            continue
        match = re.match(entity_pattern, line)
        if match:
            entity_name, entity_type = match.groups()
            entity_name = clean_entity_name(entity_name)
            entities.append((entity_name, entity_type.strip()))
    return entities

# ---- Example usage (replace with your actual entities_text) ----
entities_text = """OpenAI - Organization/Company
Google DeepMind - Organization/Company
Meta - Organization/Company
artificial general intelligence (AGI) - Goal/Concept
1956 - Year"""

entities = parse_entities(entities_text)
print(f"[OK] Extracted {len(entities)} entities. Sample:", entities[:10])


[OK] Extracted 5 entities. Sample: [('OpenAI', 'Organization/Company'), ('Google DeepMind', 'Organization/Company'), ('Meta', 'Organization/Company'), ('artificial general intelligence (AGI)', 'Goal/Concept'), ('1956', 'Year')]


In [34]:
import re

# --- tokenize a type string into lowercase tokens ---
def split_types(t: str) -> set:
    return {p.lower() for p in re.split(r'[^A-Za-z0-9]+', (t or '').strip()) if p}

# --- buckets that match your extracted types ---
APP_LIKE      = {"application","app","organization","company","product","platform","service","tool",
                 "assistant","system","software","website","search","engine","assistant"}  # e.g., YouTube, Google Search, Google Assistant
CONCEPT_LIKE  = {"concept","technology","method","technique","algorithm","model","representation",
                 "network","optimization","logic","system","type","category","acronym","research","area"}  # Method/Technology, Technology category, Product category, Research area, Acronym/Concept, System type
FIELD_LIKE    = {"field","discipline","science"}  # computer science, linguistics, psychology, economics, philosophy, neuroscience, statistics (also a method)
GOAL_LIKE     = {"goal","objective"}  # AGI often comes as Goal/Concept
TECHNIQUE_LIKE= {"method","technique","algorithm","optimization","logic","networks"}  # covers "Method", "Method/Technology"
GAME_LIKE     = {"game"}  # chess, Go
YEAR_LIKE     = {"year"}  # "1956" with type 'Year'

def create_relationship(tx, e1_name, e2_name, relationship):
    rel_clean = re.sub(r"[^A-Za-z0-9_]", "_", relationship)
    tx.run(
        "MATCH (a:Entity {name:$e1}), (b:Entity {name:$e2}) "
        f"MERGE (a)-[r:{rel_clean}]->(b)",
        e1=e1_name, e2=e2_name
    )

with driver.session() as session:
    # 1) nodes (unchanged)
    for name, etype in entities:
        session.execute_write(create_node, etype, name)

    # 2) relationships (mapped to your types)
    for i, (n1, t1) in enumerate(entities):
        T1 = split_types(t1)
        for j, (n2, t2) in enumerate(entities):
            if i == j:
                continue
            T2 = split_types(t2)

            matched = False
            def is_app_like(S):       return bool(S & APP_LIKE)
            def is_concept_like(S):   return bool(S & CONCEPT_LIKE)
            def is_field_like(S):     return bool(S & FIELD_LIKE)
            def is_goal_like(S):      return bool(S & GOAL_LIKE)
            def is_technique_like(S): return bool(S & TECHNIQUE_LIKE)
            def is_game_like(S):      return bool(S & GAME_LIKE)
            def is_year_like(S):      return bool(S & YEAR_LIKE) or (n1.isdigit() and len(n1)==4)

            # Application-like → Concept-like
            if is_app_like(T1) and is_concept_like(T2):
                session.execute_write(create_relationship, n1, n2, "USES"); matched = True

            # Application-like → Goal
            if is_app_like(T1) and is_goal_like(T2):
                session.execute_write(create_relationship, n1, n2, "ACHIEVES"); matched = True

            # Application-like → Technique
            if is_app_like(T1) and is_technique_like(T2):
                session.execute_write(create_relationship, n1, n2, "IMPLEMENTED_WITH"); matched = True

            # Game → Concept
            if is_game_like(T1) and is_concept_like(T2):
                session.execute_write(create_relationship, n1, n2, "BASED_ON"); matched = True

            # Field → Concept
            if is_field_like(T1) and is_concept_like(T2):
                session.execute_write(create_relationship, n1, n2, "STUDIES"); matched = True

            # Field → Goal
            if is_field_like(T1) and is_goal_like(T2):
                session.execute_write(create_relationship, n1, n2, "SUPPORTS"); matched = True

            # Concept → Field
            if is_concept_like(T1) and is_field_like(T2):
                session.execute_write(create_relationship, n1, n2, "APPLIES_TO"); matched = True

            # Year → Concept
            if is_year_like(T1) and is_concept_like(T2):
                session.execute_write(create_relationship, n1, n2, "INTRODUCED_IN"); matched = True

            # Goal → Technique
            if is_goal_like(T1) and is_technique_like(T2):
                session.execute_write(create_relationship, n1, n2, "ACHIEVED_BY"); matched = True

            # Concept → Technique
            if is_concept_like(T1) and is_technique_like(T2):
                session.execute_write(create_relationship, n1, n2, "ENABLED_BY"); matched = True

            # Fallback so the graph is always connected
            if not matched:
                session.execute_write(create_relationship, n1, n2, "RELATED_TO")
